# EcoFOCIpy PostQC example to update netcdf file

see [EcoFOCIpy_sbe_UAF.ipynb](EcoFOCIpy_sbe_UAF.ipynb) for initial processing and formatting.  Data would have been convertted from raw instrument format to csv files, initial dirty plots created and a meta-data rich and meta-data poor dataset would have been created.  The metadata poor csv file is used for the initial preliminary data archive, the metadata rich netcdf file is used for QC procedures and eventual final data archive.

## Next Steps

QC of data (plot parameters with other instruments)
- be sure to updated the qc_status and the history

- **TODO** Programmatically simplify following tools

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import yaml

import pandas as pd
import xarray as xr
import datetime
import EcoFOCIpy.io.erddap as erddap

import EcoFOCIpy.metaconfig.load_config as load_config

/Users/bell/src/ecofocipy/src/EcoFOCIpy/__init__.py:62: UserWarning: Could not import all EcoFOCIpy submodules: cannot import name 'instruments' from partially initialized module 'EcoFOCIpy' (most likely due to a circular import) (/Users/bell/src/ecofocipy/src/EcoFOCIpy/__init__.py)
  warnings.warn(f"Could not import all EcoFOCIpy submodules: {e}")


In [2]:
sample_data_dir = '/Users/bell/Programs/EcoFOCIpy/'
user_data_dir = '/Users/bell/ecoraid/2025/Moorings/25bsst8a/working/'
filename = '25bsst8a_s37_0027m.nc'

### Post Manual QC load data and rebuild nc file

- using excel for a few points or a dynamic web map for multiple (a tool on ecofoci-field.pmel.noaa.gov) or any other method to identify spikes
- remove values above 32psu and below 30psu

**NOTE** if you use excel, be cautious about how your time is formatted (its important) - use custom formatting and make it of the form `yyyy-mm-ddTHH:MM:SSZ` to assure proper read in to xarray

*TODO:* use erddap hosted data 

In [3]:
# this loads the initial working netcdf file
wop_nc_qc = xr.load_dataset(user_data_dir+filename)

In [4]:
wop_nc_qc

<xarray.Dataset> Size: 351kB
Dimensions:       (time: 8776, depth: 1, latitude: 1, longitude: 1)
Coordinates:
  * time          (time) datetime64[ns] 70kB 2025-08-15T05:00:00 ... 2026-08-...
  * depth         (depth) float64 8B 27.0
  * latitude      (latitude) float64 8B 62.2
  * longitude     (longitude) float64 8B -174.7
Data variables:
    temperature   (time, depth, latitude, longitude) float64 70kB 4.989 ... -...
    conductivity  (time, depth, latitude, longitude) float64 70kB 2.98 ... 2.605
    pressure      (time, depth, latitude, longitude) float64 70kB 26.94 ... 2...
    salinity      (time, depth, latitude, longitude) float64 70kB 30.78 ... 3...
Attributes: (12/33)
    Latitude_DegMMddW:                      62 12.2783 N
    Longitude_DegMMddN:                     174 40.5599 W
    MooringID:                              25BSST-8A
    platform_deployment_date:               2025-08-15T04:35:00Z
    platform_deployment_cruise_name:        SKQ202515S
    platform_recovery_date:                 2026-08-15T20:00:00Z
    ...                                     ...
    InstrumentSerialNumber:                 3764
    InstrumentType:                         SBE-37
    date_created:                           2026-09-13T10:43:39Z
    date_modified:                          
    QC_indicator:                           unknown
    history:                                Trimmed to deployment.

In [5]:
# this reads the previous csv and assumes you've made modifications to the content but not the structure (record length or variable names)
df_qc = pd.read_csv(user_data_dir+filename.replace('nc','to_edit.csv'), index_col=['time','depth','latitude','longitude']) #order is important

In [6]:
df_qc

,,,,Unnamed: 0,temperature,conductivity,pressure,salinity,salinity_rolling_mean,salinity_rolling_std,upper_bound,lower_bound
time,depth,latitude,longitude,,,,,,,,,
2025-08-15 05:00:00,27.0,62.204638,-174.675998,0,4.9894,2.97955,26.944,30.7832,30.783200,NaN,NaN,NaN
2025-08-15 06:00:00,27.0,62.204638,-174.675998,1,2.5645,2.79920,26.980,30.9507,30.866950,0.118440,31.459152,30.274748
2025-08-15 07:00:00,27.0,62.204638,-174.675998,2,3.0872,2.94055,27.122,32.1516,31.295167,0.746406,35.027199,27.563134
2025-08-15 08:00:00,27.0,62.204638,-174.675998,3,3.1706,2.90574,27.211,31.6497,31.383800,0.634696,34.557278,28.210322
2025-08-15 09:00:00,27.0,62.204638,-174.675998,4,2.2254,2.78379,27.264,31.0919,31.325420,0.564951,34.150177,28.500663
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-15 16:00:00,27.0,62.204638,-174.675998,8771,-0.3975,2.61713,26.625,31.6095,31.454029,0.130497,32.106511,30.801546
2026-08-15 17:00:00,27.0,62.204638,-174.675998,8772,-0.3405,2.60387,26.731,31.3744,31.453178,0.130547,32.105915,30.800441
2026-08-15 18:00:00,27.0,62.204638,-174.675998,8773,-0.2135,2.62418,26.891,31.5117,31.452581,0.130045,32.102807,30.802355


In [7]:
#replace original data with editied data
for var in wop_nc_qc.variables:
    if var not in ['time','latitude','longitude','depth'] + []:
        print(var)
        try:
            wop_nc_qc[var].values=xr.Dataset.from_dataframe(df_qc)[var]
            print(f'{var} success')
        except:
            pass

temperature
temperature success
conductivity
conductivity success
pressure
pressure success
salinity
salinity success


In [8]:
for var in ['latitude','longitude','depth']:
    print(var)
    try:
        wop_nc_qc.coords[var]=xr.Dataset.from_dataframe(df_qc)[var]
        print(f'{var} success')
    except:
        pass

latitude
latitude success
longitude
longitude success
depth
depth success


In [9]:
#drop oxy conc for oxygen
# wop_nc_qc = wop_nc_qc.drop('oxy_conc')

### Update global attributes

In [10]:
wop_nc_qc.attrs.update({'QC_indicator': 'ProbablyGood'})
try:
    wop_nc_qc.attrs.update({'history':(wop_nc_qc.history + "\nQC'd: "+ str(datetime.datetime.today()))})
except:
    pass

In [11]:
wop_nc_qc.attrs.update({'date_modified':str(datetime.datetime.today())})

In [12]:
wop_nc_qc.to_netcdf(user_data_dir+filename,format="NETCDF3_CLASSIC",encoding={'time':{'units':'days since 1900-01-01'}})

/var/folders/6r/n_mpcj7d7pdf9ncqvjy5vb8c0000jm/T/ipykernel_16333/664318075.py:1: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 1900-01-01'. Resolution of 'hours' needed. Serializing times to floating point instead. Set encoding['dtype'] to integer dtype to serialize to int64. Set encoding['dtype'] to floating point dtype to silence this warning.
  wop_nc_qc.to_netcdf(user_data_dir+filename,format="NETCDF3_CLASSIC",encoding={'time':{'units':'days since 1900-01-01'}})


In [13]:
wop_nc_qc

<xarray.Dataset> Size: 351kB
Dimensions:       (time: 8776, depth: 1, latitude: 1, longitude: 1)
Coordinates:
  * time          (time) datetime64[ns] 70kB 2025-08-15T05:00:00 ... 2026-08-...
  * depth         (depth) float64 8B 27.0
  * latitude      (latitude) float64 8B 62.2
  * longitude     (longitude) float64 8B -174.7
Data variables:
    temperature   (time, depth, latitude, longitude) float64 70kB 4.989 ... -...
    conductivity  (time, depth, latitude, longitude) float64 70kB 2.98 ... 2.605
    pressure      (time, depth, latitude, longitude) float64 70kB 26.94 ... 2...
    salinity      (time, depth, latitude, longitude) float64 70kB 30.78 ... 3...
Attributes: (12/33)
    Latitude_DegMMddW:                      62 12.2783 N
    Longitude_DegMMddN:                     174 40.5599 W
    MooringID:                              25BSST-8A
    platform_deployment_date:               2025-08-15T04:35:00Z
    platform_deployment_cruise_name:        SKQ202515S
    platform_recovery_date:                 2026-08-15T20:00:00Z
    ...                                     ...
    InstrumentSerialNumber:                 3764
    InstrumentType:                         SBE-37
    date_created:                           2026-09-13T10:43:39Z
    date_modified:                          2026-09-13 10:46:45.951103
    QC_indicator:                           ProbablyGood
    history:                                Trimmed to deployment.\nQC'd: 202...